AI Driver Drowsiness and Distraction Detection System.

In [1]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install mediapipe

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install mediapipe==0.10.14 "numpy<2"

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------------------- ------------ 41.0/61.0 kB 245.8 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 295.3 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of ml-dtypes to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   --------------------------------

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import csv
import os
import time
import threading
from datetime import datetime
import cv2
import numpy as np
import mediapipe as mp

# Windows built-in sound; no separate audio file or Pygame required.
try:
    import winsound
except ImportError:
    winsound = None

# ---------------- SETTINGS ----------------
CALIBRATION_SECS = 5
EYE_CLOSED_SECS = 1.5
YAWN_SECS = 1.2
HEAD_DOWN_SECS = 2.0
LOOK_AWAY_SECS = 1.5

EAR_FACTOR = 0.70
MAR_YAWN = 0.60
HEAD_DELTA = 0.15
LOOK_AWAY_RATIO = 0.18
ALARM_COOLDOWN = 1.5
NO_FACE_SECS = 2.0              # face missing this long -> alarm
LOG_FILE = "alerts_log.csv"     # every alert is saved here

# MediaPipe Face Mesh landmark indices
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
LEFT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [13, 14, 78, 308]
NOSE = 1
CHIN = 152
LEFT_EYE_CORNER = 33
RIGHT_EYE_CORNER = 263


def points(landmarks, indices, width, height):
    """Convert normalized landmarks to pixel coordinates."""
    return np.array(
        [(landmarks[i].x * width, landmarks[i].y * height)
         for i in indices],
        dtype=np.float32
    )


def eye_aspect_ratio(p):
    """Estimate eye openness; lower values usually mean the eye is closed."""
    vertical_1 = np.linalg.norm(p[1] - p[5])
    vertical_2 = np.linalg.norm(p[2] - p[4])
    horizontal = np.linalg.norm(p[0] - p[3])
    if horizontal == 0:
        return 0.0
    return (vertical_1 + vertical_2) / (2.0 * horizontal)


def mouth_aspect_ratio(p):
    """Estimate mouth opening relative to mouth width."""
    width = np.linalg.norm(p[2] - p[3])
    if width == 0:
        return 0.0
    return np.linalg.norm(p[0] - p[1]) / width


def normalized_head_ratio(landmarks, height):
    """Heuristic based on nose position between the eye line and chin."""
    nose_y = landmarks[NOSE].y * height
    eyes_y = ((landmarks[33].y + landmarks[263].y) / 2.0) * height
    chin_y = landmarks[CHIN].y * height
    distance = chin_y - eyes_y
    if abs(distance) < 1:
        return 0.0
    return (nose_y - eyes_y) / distance


def play_alarm():
    """Play a short beep on Windows without blocking video processing."""
    if winsound is not None:
        threading.Thread(
            target=lambda: winsound.Beep(1000, 350),
            daemon=True
        ).start()
    else:
        print("\aWARNING: possible drowsiness or distraction!")


def log_event(alert_type):
    """Append one row (timestamp, alert type) to the CSV log."""
    new_file = not os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["timestamp", "alert_type"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                         alert_type])


def main():
    camera = cv2.VideoCapture(0)
    if not camera.isOpened():
        print("ERROR: Could not open webcam.")
        print("Check Windows camera permissions and close other apps using it.")
        return

    calibration_start = None      # set when the face is first seen
    no_face_start = None
    last_logged_warning = ""
    calibration_ears = []
    calibration_heads = []
    calibration_noses = []

    baseline_ear = None
    baseline_head = None
    baseline_nose_offset = None

    eye_start = None
    yawn_start = None
    head_start = None
    look_start = None
    last_alarm_time = 0.0

    face_mesh = mp.solutions.face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    print("Driver Safety AI started. Press Q in the camera window to quit.")

    try:
        while True:
            success, frame = camera.read()
            if not success:
                print("ERROR: Could not read a webcam frame.")
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            result = face_mesh.process(rgb_frame)

            now = time.time()
            status = "NORMAL"
            warning = ""
            color = (0, 200, 0)

            if not result.multi_face_landmarks:
                status = "NO FACE"
                color = (0, 165, 255)
                cv2.putText(
                    frame, "No face detected", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2
                )
                eye_start = yawn_start = head_start = look_start = None

                # Face missing too long (head dropped / turned away) -> alarm
                if baseline_ear is not None:
                    no_face_start = no_face_start or now
                    if now - no_face_start >= NO_FACE_SECS:
                        warning = "FACE NOT VISIBLE: EYES ON ROAD!"

            else:
                no_face_start = None
                landmarks = result.multi_face_landmarks[0].landmark

                right_ear = eye_aspect_ratio(
                    points(landmarks, RIGHT_EYE, width, height)
                )
                left_ear = eye_aspect_ratio(
                    points(landmarks, LEFT_EYE, width, height)
                )
                ear = (right_ear + left_ear) / 2.0

                mar = mouth_aspect_ratio(
                    points(landmarks, MOUTH, width, height)
                )
                head_ratio = normalized_head_ratio(landmarks, height)

                # Approximate head direction using nose offset between eye corners.
                left_x = landmarks[LEFT_EYE_CORNER].x * width
                right_x = landmarks[RIGHT_EYE_CORNER].x * width
                nose_x = landmarks[NOSE].x * width
                eye_width = abs(right_x - left_x)

                if eye_width > 1:
                    nose_offset = (
                        nose_x - (left_x + right_x) / 2.0
                    ) / eye_width
                else:
                    nose_offset = 0.0

                # ----- Five-second calibration -----
                if baseline_ear is None:
                    if calibration_start is None:
                        calibration_start = now
                    calibration_ears.append(ear)
                    calibration_heads.append(head_ratio)
                    calibration_noses.append(nose_offset)

                    seconds_left = max(
                        0, CALIBRATION_SECS - (now - calibration_start)
                    )
                    cv2.putText(
                        frame,
                        f"Calibrating: look straight, eyes open ({seconds_left:.0f}s)",
                        (20, 35), cv2.FONT_HERSHEY_SIMPLEX,
                        0.62, (255, 255, 0), 2
                    )

                    if now - calibration_start >= CALIBRATION_SECS and calibration_ears:
                        baseline_ear = float(np.median(calibration_ears))
                        baseline_head = float(np.median(calibration_heads))
                        baseline_nose_offset = float(np.median(calibration_noses))
                        if baseline_ear <= 0:
                            baseline_ear = 0.25

                else:
                    # 1. Eyes closed for too long
                    if ear < baseline_ear * EAR_FACTOR:
                        eye_start = eye_start or now
                        if now - eye_start >= EYE_CLOSED_SECS:
                            warning = "DROWSY: EYES CLOSED"
                    else:
                        eye_start = None

                    # 2. Mouth open for a sustained period (possible yawn)
                    if mar > MAR_YAWN:
                        yawn_start = yawn_start or now
                        if now - yawn_start >= YAWN_SECS and not warning:
                            warning = "POSSIBLE YAWN: TAKE A BREAK"
                    else:
                        yawn_start = None

                    # 3. Head/nose position changed from calibration
                    if abs(head_ratio - baseline_head) > HEAD_DELTA:
                        head_start = head_start or now
                        if now - head_start >= HEAD_DOWN_SECS and not warning:
                            warning = "CHECK HEAD POSITION"
                    else:
                        head_start = None

                    # 4. Approximate head turn; this is not precise eye-gaze detection.
                    if abs(nose_offset - baseline_nose_offset) > LOOK_AWAY_RATIO:
                        look_start = look_start or now
                        if now - look_start >= LOOK_AWAY_SECS and not warning:
                            warning = "DISTRACTED: LOOK FORWARD"
                    else:
                        look_start = None

                    if warning:
                        status = warning
                        color = (0, 0, 255)

                    cv2.putText(
                        frame, f"EAR: {ear:.2f}   MAR: {mar:.2f}",
                        (20, 30), cv2.FONT_HERSHEY_SIMPLEX,
                        0.65, (0, 255, 0), 2
                    )

                cv2.putText(
                    frame, f"STATUS: {status}", (20, height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2
                )

            # ----- Alarm, red border and logging (runs for every warning) -----
            if warning:
                cv2.rectangle(
                    frame, (0, 0), (width - 1, height - 1),
                    (0, 0, 255), 8
                )
                cv2.putText(
                    frame, warning, (20, height - 55),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2
                )
                if now - last_alarm_time >= ALARM_COOLDOWN:
                    play_alarm()
                    last_alarm_time = now
                if warning != last_logged_warning:   # log once per new alert
                    log_event(warning)
                    last_logged_warning = warning
            else:
                last_logged_warning = ""

            cv2.imshow("Driver Drowsiness and Distraction Detection", frame)
            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), ord("Q")):
                break

    finally:
        camera.release()
        face_mesh.close()
        cv2.destroyAllWindows()
        print("Camera closed. Program stopped.")


if __name__ == "__main__":
    main()

Driver Safety AI started. Press Q in the camera window to quit.
Camera closed. Program stopped.
